# Fireworks Serverless RL — explicit Colab walkthrough

This notebook is a line-by-line teaching adaptation of the Fireworks cookbook Countdown example. It keeps the upstream reward function, GRPO filtering, importance-sampling update, Router Replay, evaluation artifacts, and optional W&B logging, while defaulting to a small smoke-test profile.

**Cost boundary:** setup, data preparation, and reward tests are local. Fireworks API usage begins only in the final run cell, and that cell is disabled until you set `RUN_TRAINING = True`. A Colab GPU is not required because training and sampling run on Fireworks.

## 1. Install the tested cookbook training package

The checkout is pinned to the exact upstream revision tested with this notebook. The stable Fireworks SDK required by the cookbook is installed by the package, so `--pre` is intentionally not used. The next cell adds the checkout to `sys.path`, so no kernel restart is needed after this install.

In [ ]:
%%bash
set -euo pipefail
COOKBOOK_COMMIT=8c8366e7d70618bccef9047b32c4e982a9df2eef
if [ ! -d /content/fw-cookbook/.git ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/fw-ai/cookbook.git /content/fw-cookbook
fi
cd /content/fw-cookbook
git fetch --depth 1 origin "$COOKBOOK_COMMIT"
git checkout --detach "$COOKBOOK_COMMIT"
git sparse-checkout set training
python -m pip install -q -e training

## 2. Load secrets

In Colab, open the key icon (**Secrets**) and add `FIREWORKS_API_KEY`. W&B is opt-in: add both `WANDB_API_KEY` and `WANDB_ENTITY`. Use `fireworks-ai-llm` for the Fireworks team or `sophia-yang` for your personal account. Never paste either key into a notebook cell.

In [ ]:
import os

def colab_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        return userdata.get(name) or default
    except Exception:
        return os.environ.get(name, default)

FIREWORKS_API_KEY = colab_secret("FIREWORKS_API_KEY")
WANDB_API_KEY = colab_secret("WANDB_API_KEY")
WANDB_ENTITY = colab_secret("WANDB_ENTITY")
WANDB_PROJECT = "serverless-rl-countdown"

if FIREWORKS_API_KEY:
    os.environ["FIREWORKS_API_KEY"] = FIREWORKS_API_KEY
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TRUST_REMOTE_CODE"] = "1"

print("Fireworks key found:", bool(FIREWORKS_API_KEY))
print("W&B logging enabled:", bool(WANDB_API_KEY and WANDB_ENTITY))
print("W&B entity:", WANDB_ENTITY)

## 3. Choose data and hyperparameters

Choose `smoke` first. It uses the bundled sample and a small number of short rollouts. `full` matches the upstream 20-step defaults and prepares the deterministic 20,000-row TinyZero subset. Changing the profile does not start paid work; the separate paid-run gate remains off by default.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

COOKBOOK = Path("/content/fw-cookbook")
COOKBOOK_COMMIT = "8c8366e7d70618bccef9047b32c4e982a9df2eef"
EXAMPLE_DIR = COOKBOOK / "training/examples/serverless_rl"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path("/content/serverless_rl_runs") / RUN_STAMP
RUN_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = "smoke"  # change to "full" only after the smoke run succeeds
if RUN_PROFILE not in {"smoke", "full"}:
    raise ValueError("RUN_PROFILE must be 'smoke' or 'full'")

USE_CANONICAL_DATASET = RUN_PROFILE == "full"
PREPARE_DATASET_ROWS = 20_000
SEED = 0
EVAL_SEED = 1
DATASET_PATH = EXAMPLE_DIR / "data/countdown_3to4_train.jsonl" if USE_CANONICAL_DATASET else EXAMPLE_DIR / "data/countdown_train.jsonl"

BASE_MODEL = "accounts/fireworks/models/kimi-k3"
TOKENIZER_MODEL = "moonshotai/Kimi-K3"
RENDERER_NAME = ""  # blank means auto-resolve
LORA_RANK = 32
LORA_ALPHA = 64
MAX_SEQ_LEN = 32_768
STEPS = 1 if RUN_PROFILE == "smoke" else 20
PROMPT_GROUPS_PER_STEP = 2 if RUN_PROFILE == "smoke" else 16
GROUP_SIZE = 2 if RUN_PROFILE == "smoke" else 8
PROMPT_CONCURRENCY = 2 if RUN_PROFILE == "smoke" else 8
MAX_SAMPLE_TOKENS = 512 if RUN_PROFILE == "smoke" else 4_096
TEMPERATURE = 1.0
LEARNING_RATE = 1e-4
ROUTER_REPLAY = True
EVAL_PROMPT_GROUPS = 2 if RUN_PROFILE == "smoke" else 16
EVAL_GROUP_SIZE = 2 if RUN_PROFILE == "smoke" else 8
EVAL_INTERVAL = 0 if RUN_PROFILE == "smoke" else 5
DCP_SAVE_INTERVAL = 0 if RUN_PROFILE == "smoke" else 5
DCP_TIMEOUT_S = 900
RESUME_FROM = ""  # optional: account/run-id/cd-state-NNNN
SAMPLING_TIMEOUT_S = 1_800
API_BASE_URL = os.environ.get("FIREWORKS_BASE_URL", "https://api.fireworks.ai")

print(f"profile={RUN_PROFILE} run_dir={RUN_DIR}")
print(f"planned training completions={STEPS * PROMPT_GROUPS_PER_STEP * GROUP_SIZE:,}")

In [ ]:
import hashlib
import subprocess
import sys

# An editable install is not picked up by an already-running kernel.
if str(COOKBOOK) not in sys.path:
    sys.path.insert(0, str(COOKBOOK))

from training.examples.serverless_rl.countdown_rl import prepare_dataset

if USE_CANONICAL_DATASET:
    existing_rows = sum(1 for line in DATASET_PATH.open() if line.strip()) if DATASET_PATH.exists() else 0
    if existing_rows != PREPARE_DATASET_ROWS:
        print(f"Preparing canonical dataset ({existing_rows:,} usable rows found; expected {PREPARE_DATASET_ROWS:,}).")
        prepare_dataset(DATASET_PATH, PREPARE_DATASET_ROWS, SEED)
elif not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

dataset_rows = sum(1 for line in DATASET_PATH.open() if line.strip())
dataset_sha256 = hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest()
checked_out_commit = subprocess.check_output(["git", "-C", str(COOKBOOK), "rev-parse", "HEAD"], text=True).strip()
if checked_out_commit != COOKBOOK_COMMIT:
    raise RuntimeError(f"Expected cookbook {COOKBOOK_COMMIT}, found {checked_out_commit}")

print("Dataset:", DATASET_PATH)
print("Rows:", dataset_rows)
print("Dataset SHA-256:", dataset_sha256)
print("Cookbook commit:", checked_out_commit)

## 4. Reward function, exactly matching upstream

The model earns 0.1 for an answer tag, 0.2 for using exactly the supplied numbers, and 0.7 for reaching the target. Exponentiation (`**`) and floor division (`//`) are explicitly rejected because the task allows only `+ - * /`.

In [ ]:
from __future__ import annotations

import json
import math
import re

from training.examples.serverless_rl.countdown_rewards import composite_reward as upstream_composite_reward

def extract_answer(text: str) -> str | None:
    matches = list(re.finditer(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL))
    return matches[-1].group(1).strip() if matches else None

def safe_eval_equation(equation: str) -> float | None:
    cleaned = equation.replace(" ", "")
    if not re.match(r"^[\d+\-*/().]+$", cleaned):
        return None
    if "**" in cleaned or "//" in cleaned:
        return None
    try:
        return float(eval(cleaned, {"__builtins__": None}, {}))
    except Exception:
        return None

def parse_ground_truth(ground_truth: str | dict) -> tuple[list[int], int]:
    gt = json.loads(ground_truth) if isinstance(ground_truth, str) else ground_truth
    numbers = gt.get("numbers") or gt.get("nums")
    if numbers is None:
        raise KeyError("ground_truth must include 'numbers' or 'nums'")
    return list(numbers), int(gt["target"])

def check_numbers_used(equation: str, numbers: list[int]) -> bool:
    return sorted(int(x) for x in re.findall(r"\d+", equation)) == sorted(numbers)

def composite_reward(response: str, ground_truth: str | dict) -> float:
    numbers, target = parse_ground_truth(ground_truth)
    equation = extract_answer(response)
    if equation is None:
        return 0.0
    score = 0.1
    numbers_valid = check_numbers_used(equation, numbers)
    if numbers_valid:
        score += 0.2
    result = safe_eval_equation(equation)
    if numbers_valid and result is not None and abs(result - target) < 1e-6:
        score += 0.7
    return score

gt = {"numbers": [1, 2, 3], "target": 6}
assert math.isclose(composite_reward("<think>multiply</think><answer>1 * 2 * 3</answer>", gt), 1.0)
assert math.isclose(composite_reward("<answer>1 + 2 + 3</answer>", {**gt, "target": 7}), 0.3)
assert math.isclose(composite_reward("<answer>1 + 2</answer>", gt), 0.1)
assert safe_eval_equation("2 ** 3") is None
assert safe_eval_equation("6 // 2") is None
reward_cases = [
    ("<think>multiply</think><answer>1 * 2 * 3</answer>", gt),
    ("<answer>1 + 2</answer>", gt),
    ("no answer tag", gt),
]
for response, ground_truth in reward_cases:
    assert math.isclose(composite_reward(response, ground_truth), upstream_composite_reward(response, ground_truth))
print("Reward tests and upstream parity checks passed. Fully correct score:", composite_reward("<answer>1 * 2 * 3</answer>", gt))

## 5. Make a fixed, disjoint evaluation split

The same held-out prompts are reused throughout the run. Their row indices are saved so the split is reproducible.

In [ ]:
import random

all_rows = [json.loads(line) for line in DATASET_PATH.open() if line.strip()]
if EVAL_PROMPT_GROUPS >= len(all_rows):
    raise ValueError("EVAL_PROMPT_GROUPS must leave at least one training row")
indices = list(range(len(all_rows)))
random.Random(EVAL_SEED).shuffle(indices)
eval_indices = indices[:EVAL_PROMPT_GROUPS]
eval_index_set = set(eval_indices)
train_rows = [row for i, row in enumerate(all_rows) if i not in eval_index_set]
eval_rows = [all_rows[i] for i in eval_indices]
split_record = {
    "dataset": str(DATASET_PATH), "total_rows": len(all_rows),
    "training_rows": len(train_rows), "eval_rows": len(eval_rows),
    "eval_seed": EVAL_SEED, "eval_row_indices": eval_indices,
    "dataset_sha256": dataset_sha256, "cookbook_commit": checked_out_commit,
    "run_profile": RUN_PROFILE,
}
(RUN_DIR / "dataset_split.json").write_text(json.dumps(split_record, indent=2) + "\n")
order = list(range(len(train_rows)))
random.Random(SEED).shuffle(order)
print(f"train={len(train_rows):,} eval={len(eval_rows):,}")

## 6. Load the model-specific tokenizer and renderer

The renderer turns chat messages into model tokens and parses sampled tokens back into assistant text. Helper functions are imported from the checked-out upstream example so URL normalization, GRPO math, truncation detection, KLD diagnostics, and Router Replay stay synchronized with the cookbook.

In [ ]:
import math
import time
from typing import Any

import tinker
from fireworks.training.sdk import FireworksClient, FiretitanSamplingParams, FiretitanServiceClient
from training.examples.serverless_rl.countdown_rl import (
    _account_from_session, _control_plane_base_url, _finish_reason,
    _group_relative_advantages, _step_from_checkpoint_name,
    _is_truncated, _mean_loss, _mean_policy_sample_k3,
    _mean_policy_sample_logprob_gap, _serverless_base_url,
)
from training.renderer import get_renderer, get_text_content
import training.renderer  # registers cookbook-local renderers
from training.utils import resolve_router_replay_enabled
from training.utils.rl.router_replay import build_r3_routing_matrices, validate_r3_routing_matrices
from training.utils.supervised import resolve_renderer_name
from training.utils.tokenizers import load_tokenizer

tokenizer = load_tokenizer(TOKENIZER_MODEL)
resolved_renderer_name = resolve_renderer_name(TOKENIZER_MODEL, RENDERER_NAME)
renderer = get_renderer(resolved_renderer_name, tokenizer)
print("Renderer:", resolved_renderer_name)
print("Serverless URL:", _serverless_base_url(API_BASE_URL))
print("Control-plane URL:", _control_plane_base_url(API_BASE_URL))

## 7. The training step, separated into four inspectable stages

1. Save the current adapter and sample groups.
2. Decode and score every valid completion.
3. Drop zero-variance groups, compute group-relative advantages, and build aligned training datums.
4. Run one importance-sampling forward/backward pass and one Adam step.

In [ ]:
def validate_length(label: str, length: int) -> None:
    if length > MAX_SEQ_LEN:
        raise ValueError(f"{label} length {length} exceeds MAX_SEQ_LEN={MAX_SEQ_LEN}")

def stage1_snapshot_and_sample(training_client, service, batch, step, router_replay_enabled):
    snapshot = training_client.save_weights_for_sampler(f"cd-sample-{step:04d}").result().path
    if not snapshot:
        raise RuntimeError("save_weights_for_sampler returned no path")
    prompts = [renderer.build_generation_prompt(row["messages"]) for row in batch]
    for prompt in prompts:
        validate_length("prompt + max sample tokens", prompt.length + MAX_SAMPLE_TOKENS)
    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    try:
        params = FiretitanSamplingParams(
            max_tokens=MAX_SAMPLE_TOKENS, temperature=TEMPERATURE,
            stop=renderer.get_stop_sequences(),
            include_routing_matrix=router_replay_enabled,
        )
        results = []
        for start in range(0, len(prompts), PROMPT_CONCURRENCY):
            futures = [sampler.sample(prompt=p, num_samples=GROUP_SIZE, sampling_params=params)
                       for p in prompts[start:start + PROMPT_CONCURRENCY]]
            results.extend(f.result(timeout=SAMPLING_TIMEOUT_S) for f in futures)
    finally:
        sampler.close()
    return snapshot, prompts, results

In [ ]:
def stage2_decode_and_score(results, prompts, batch):
    groups, raw_rewards = [], []
    response_tokens = truncated = max_response_tokens = 0
    for result, prompt, row in zip(results, prompts, batch, strict=True):
        group = {"prompt": prompt, "row": row, "samples": []}
        for seq in getattr(result, "sequences", []) or []:
            tokens = list(getattr(seq, "tokens", []) or [])
            logprobs = getattr(seq, "logprobs", None)
            if not tokens or logprobs is None or len(logprobs) != len(tokens):
                continue
            text = get_text_content(renderer.parse_response(tokens)[0])
            reward = float(composite_reward(text, row["ground_truth"]))
            routes = getattr(seq, "routing_matrices", None)
            group["samples"].append({
                "tokens": tokens, "logprobs": [float(x) for x in logprobs],
                "routes": list(routes) if routes is not None else None,
                "reward": reward, "text": text,
            })
            raw_rewards.append(reward)
            response_tokens += len(tokens)
            max_response_tokens = max(max_response_tokens, len(tokens))
            truncated += int(_is_truncated(seq, MAX_SAMPLE_TOKENS))
        groups.append(group)
    return groups, {
        "raw_rewards": raw_rewards, "response_tokens": response_tokens,
        "max_response_tokens": max_response_tokens, "truncated": truncated,
    }

In [ ]:
def stage3_build_datums(groups, router_replay_enabled):
    datums, filtered_rewards = [], []
    for group in groups:
        rewards = [sample["reward"] for sample in group["samples"]]
        if len(set(rewards)) <= 1:
            continue  # no within-group learning signal
        filtered_rewards.extend(rewards)
        advantages = _group_relative_advantages(rewards)
        prompt = group["prompt"]
        response_start = prompt.length - 1
        for sample, advantage in zip(group["samples"], advantages, strict=True):
            tokens = sample["tokens"]
            model_input = prompt.append(tinker.EncodedTextChunk(tokens=tokens[:-1]))
            validate_length("training datum", model_input.length)
            if router_replay_enabled:
                validate_r3_routing_matrices(sample["routes"], prompt_len=prompt.length, model_input_len=model_input.length)
                aligned = build_r3_routing_matrices(
                    sample["routes"], prompt_len=prompt.length,
                    model_input_len=model_input.length, completion_only=True,
                )
                model_input = model_input.model_copy(update={"routing_matrices": aligned})
            response_len = model_input.length - response_start
            datums.append(tinker.Datum(model_input=model_input, loss_fn_inputs={
                "target_tokens": [0] * response_start + tokens,
                "logprobs": [0.0] * response_start + sample["logprobs"],
                "advantages": [0.0] * response_start + [advantage] * response_len,
            }))
    return datums, filtered_rewards

In [ ]:
def stage4_update(training_client, datums):
    if not datums:
        return None, None, None
    fb = training_client.forward_backward(datums, "importance_sampling").result()
    loss = _mean_loss(fb)
    k1 = _mean_policy_sample_logprob_gap(datums, fb)
    k3 = _mean_policy_sample_k3(datums, fb)
    adam = tinker.AdamParams(
        learning_rate=LEARNING_RATE, beta1=0.9, beta2=0.95,
        eps=1e-12, weight_decay=0.0,
    )
    training_client.optim_step(adam).result()
    return loss, k1, k3

def training_step(training_client, service, batch, step, router_replay_enabled):
    started = time.time()
    snapshot, prompts, results = stage1_snapshot_and_sample(training_client, service, batch, step, router_replay_enabled)
    groups, stats = stage2_decode_and_score(results, prompts, batch)
    datums, filtered_rewards = stage3_build_datums(groups, router_replay_enabled)
    loss, k1, k3 = stage4_update(training_client, datums)
    raw = stats["raw_rewards"]
    rec = {
        "step": step, "snapshot": snapshot,
        "rollout/raw_reward": sum(raw) / len(raw) if raw else 0.0,
        "rollout/filtered_reward": sum(filtered_rewards) / len(filtered_rewards) if filtered_rewards else 0.0,
        "rollout/raw_samples": len(raw), "rollout/filtered_samples": len(filtered_rewards),
        "rollout/filter_ratio": 1 - len(filtered_rewards) / len(raw) if raw else 0.0,
        "rollout/mean_response_tokens": stats["response_tokens"] / len(raw) if raw else 0.0,
        "rollout/max_response_tokens": stats["max_response_tokens"],
        "rollout/truncated_samples": stats["truncated"],
        "rollout/truncation_ratio": stats["truncated"] / len(raw) if raw else 0.0,
        "train/loss": loss, "train/trained": bool(datums),
        "train/router_replay": router_replay_enabled,
        "train/inference_k1": k1, "train/inference_k3": k3,
        "kld/mean_k1": k1, "kld/mean_k3": k3,
        "perf/step_wall_time": time.time() - started,
    }
    return rec

## 8. Fixed held-out evaluation

Evaluation saves its own sampler snapshot but never calls backward or Adam. In addition to aggregate metrics, every decoded completion is saved for inspection.

In [ ]:
def evaluate(training_client, service, completed_steps):
    started = time.time()
    snapshot = training_client.save_weights_for_sampler(f"cd-eval-{completed_steps:04d}").result().path
    if not snapshot:
        raise RuntimeError(f"Evaluation snapshot at step {completed_steps} returned no path")
    prompts = [renderer.build_generation_prompt(row["messages"]) for row in eval_rows]
    for prompt in prompts:
        validate_length("eval prompt + max sample tokens", prompt.length + MAX_SAMPLE_TOKENS)
    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    try:
        params = FiretitanSamplingParams(max_tokens=MAX_SAMPLE_TOKENS, temperature=TEMPERATURE, stop=renderer.get_stop_sequences())
        results = []
        for start in range(0, len(prompts), PROMPT_CONCURRENCY):
            futures = [sampler.sample(prompt=p, num_samples=EVAL_GROUP_SIZE, sampling_params=params)
                       for p in prompts[start:start + PROMPT_CONCURRENCY]]
            results.extend(f.result(timeout=SAMPLING_TIMEOUT_S) for f in futures)
    finally:
        sampler.close()
    rewards, lengths, completions = [], [], []
    truncated = 0
    for row_index, row, result in zip(eval_indices, eval_rows, results, strict=True):
        numbers, target = parse_ground_truth(row["ground_truth"])
        for sample_index, seq in enumerate(getattr(result, "sequences", []) or []):
            tokens = list(getattr(seq, "tokens", []) or [])
            text = get_text_content(renderer.parse_response(tokens)[0])
            reward = float(composite_reward(text, row["ground_truth"]))
            rewards.append(reward); lengths.append(len(tokens))
            truncated += int(_is_truncated(seq, MAX_SAMPLE_TOKENS))
            completions.append({
                "completed_steps": completed_steps, "snapshot": snapshot,
                "dataset_row_index": row_index, "numbers": numbers, "target": target,
                "sample_index": sample_index, "reward": reward,
                "response_tokens": len(tokens), "finish_reason": _finish_reason(seq),
                "response": text,
            })
    expected = len(eval_rows) * EVAL_GROUP_SIZE
    returned = len(rewards)
    rec = {
        "completed_steps": completed_steps, "snapshot": snapshot,
        "eval/raw_reward": sum(rewards) / returned if returned else 0.0,
        "eval/prompt_groups": len(eval_rows), "eval/samples": returned,
        "eval/expected_samples": expected, "eval/return_ratio": returned / expected if expected else 0.0,
        "eval/mean_response_tokens": sum(lengths) / returned if returned else 0.0,
        "eval/max_response_tokens": max(lengths, default=0),
        "eval/truncated_samples": truncated,
        "eval/truncation_ratio": truncated / returned if returned else 0.0,
        "perf/eval_wall_time": time.time() - started,
    }
    completion_dir = RUN_DIR / "eval_completions"
    completion_dir.mkdir(exist_ok=True)
    with (completion_dir / f"step-{completed_steps:04d}.jsonl").open("w") as handle:
        for item in completions:
            handle.write(json.dumps(item) + "\n")
    return rec

## 9. Explicit paid-run gate

Leave this `False` while reading or testing setup. Change it to `True` only when you intend to create a Fireworks serverless training session and incur usage. W&B is optional and activates only when both its key and entity are present.

In [ ]:
RUN_TRAINING = False  # <-- change only when ready to incur Fireworks usage

In [ ]:
def append_jsonl(path: Path, record: dict) -> None:
    with path.open("a") as handle:
        handle.write(json.dumps(record) + "\n")

def validate_run_config() -> None:
    errors = []
    for name, value in {
        "STEPS": STEPS, "PROMPT_GROUPS_PER_STEP": PROMPT_GROUPS_PER_STEP,
        "GROUP_SIZE": GROUP_SIZE, "PROMPT_CONCURRENCY": PROMPT_CONCURRENCY,
        "MAX_SAMPLE_TOKENS": MAX_SAMPLE_TOKENS, "MAX_SEQ_LEN": MAX_SEQ_LEN,
        "EVAL_PROMPT_GROUPS": EVAL_PROMPT_GROUPS, "EVAL_GROUP_SIZE": EVAL_GROUP_SIZE,
    }.items():
        if value <= 0:
            errors.append(f"{name} must be > 0")
    if GROUP_SIZE < 2:
        errors.append("GROUP_SIZE must be >= 2 to produce group-relative advantages")
    if EVAL_INTERVAL < 0 or DCP_SAVE_INTERVAL < 0:
        errors.append("EVAL_INTERVAL and DCP_SAVE_INTERVAL must be >= 0")
    if EVAL_PROMPT_GROUPS >= len(all_rows):
        errors.append("EVAL_PROMPT_GROUPS must leave at least one training row")
    if MAX_SAMPLE_TOKENS >= MAX_SEQ_LEN:
        errors.append("MAX_SAMPLE_TOKENS must be smaller than MAX_SEQ_LEN")
    if not DATASET_PATH.exists() or not train_rows:
        errors.append("the prepared training dataset is missing or empty")
    if errors:
        raise ValueError("Invalid run configuration:\n- " + "\n- ".join(errors))

def wandb_log(run, record, completed_steps):
    if run is None:
        return
    try:
        payload = {"train/step": completed_steps}
        payload.update({k: v for k, v in record.items() if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v))})
        run.log(payload)
    except Exception as exc:
        print(f"W&B log skipped: {exc}", flush=True)

def init_wandb(active_base_model, start_step):
    if not (WANDB_API_KEY and WANDB_ENTITY):
        print("W&B disabled (missing WANDB_API_KEY or WANDB_ENTITY).")
        return None
    try:
        import wandb
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, config={
            "run_profile": RUN_PROFILE, "cookbook_commit": checked_out_commit,
            "dataset_sha256": dataset_sha256, "base_model": active_base_model,
            "tokenizer_model": TOKENIZER_MODEL, "renderer_name": resolved_renderer_name,
            "dataset": str(DATASET_PATH), "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
            "max_seq_len": MAX_SEQ_LEN, "steps": STEPS, "start_step": start_step,
            "prompt_groups_per_step": PROMPT_GROUPS_PER_STEP, "group_size": GROUP_SIZE,
            "max_sample_tokens": MAX_SAMPLE_TOKENS, "temperature": TEMPERATURE,
            "learning_rate": LEARNING_RATE, "router_replay": ROUTER_REPLAY,
            "eval_prompt_groups": EVAL_PROMPT_GROUPS, "eval_group_size": EVAL_GROUP_SIZE,
            "eval_interval": EVAL_INTERVAL, "dcp_save_interval": DCP_SAVE_INTERVAL,
            "resume_from": RESUME_FROM or None, "loss": "importance_sampling",
        })
        for prefix in ("rollout/*", "train/*", "kld/*", "eval/*", "perf/*"):
            wandb.define_metric(prefix, step_metric="train/step")
        print("W&B run:", run.url)
        return run
    except Exception as exc:
        try:
            if 'wandb' in locals() and wandb.run is not None:
                wandb.finish()
        except Exception:
            pass
        print(f"W&B initialization skipped: {exc}", flush=True)
        return None

def save_dcp_checkpoint(training_client, completed_step, session_name, run_id):
    name = f"cd-state-{completed_step:04d}"
    training_client.save_state(name).result(timeout=DCP_TIMEOUT_S)
    if not run_id:
        raise RuntimeError("Training client did not expose run_id; cannot build resume reference")
    account = _account_from_session(session_name)
    if account is None:
        cp_client = FireworksClient(api_key=FIREWORKS_API_KEY, base_url=_control_plane_base_url(API_BASE_URL))
        try:
            account = cp_client.account_id
        finally:
            cp_client.close()
    reference = f"{account}/{run_id}/{name}"
    (RUN_DIR / "resume_from.txt").write_text(reference + "\n")
    print("Resume checkpoint:", reference, flush=True)
    return reference

In [ ]:
service = None
wandb_run = None
if not RUN_TRAINING:
    print("Paid run skipped. Set RUN_TRAINING = True in the previous cell when ready.")
else:
    if not FIREWORKS_API_KEY:
        raise RuntimeError("Add FIREWORKS_API_KEY to Colab Secrets first.")
    validate_run_config()
    run_config = {
        "run_profile": RUN_PROFILE, "cookbook_commit": checked_out_commit,
        "dataset": str(DATASET_PATH), "dataset_sha256": dataset_sha256,
        "base_model": BASE_MODEL, "tokenizer_model": TOKENIZER_MODEL,
        "renderer_name": resolved_renderer_name, "steps": STEPS,
        "prompt_groups_per_step": PROMPT_GROUPS_PER_STEP, "group_size": GROUP_SIZE,
        "max_sample_tokens": MAX_SAMPLE_TOKENS, "temperature": TEMPERATURE,
        "learning_rate": LEARNING_RATE, "eval_prompt_groups": EVAL_PROMPT_GROUPS,
        "eval_group_size": EVAL_GROUP_SIZE, "eval_interval": EVAL_INTERVAL,
        "dcp_save_interval": DCP_SAVE_INTERVAL, "resume_from": RESUME_FROM or None,
    }
    (RUN_DIR / "config.json").write_text(json.dumps(run_config, indent=2) + "\n")
    (RUN_DIR / "cookbook_commit.txt").write_text(checked_out_commit + "\n")
    (RUN_DIR / "input_sha256.txt").write_text(f"{dataset_sha256}  {DATASET_PATH.name}\n")
    metrics_path = RUN_DIR / "metrics.jsonl"
    eval_metrics_path = RUN_DIR / "eval_metrics.jsonl"
    metrics_path.unlink(missing_ok=True)
    eval_metrics_path.unlink(missing_ok=True)
    evaluated_at = set()
    saved_dcp_steps = set()
    latest_resume_reference = RESUME_FROM
    try:
        service = FiretitanServiceClient(api_key=FIREWORKS_API_KEY, base_url=_serverless_base_url(API_BASE_URL))
        start_step = 0
        active_base_model = BASE_MODEL
        if RESUME_FROM:
            training_client = service.create_training_client_from_state_with_optimizer(RESUME_FROM)
            start_step = _step_from_checkpoint_name(RESUME_FROM)
            weights_info = service.create_rest_client().get_weights_info_by_tinker_path(RESUME_FROM).result()
            active_base_model = weights_info.base_model
            print(f"Resuming from step {start_step}: {RESUME_FROM}", flush=True)
        else:
            training_client = service.create_lora_training_client(
                base_model=BASE_MODEL, rank=LORA_RANK, alpha=LORA_ALPHA,
            )
        router_replay_enabled = resolve_router_replay_enabled(
            requested=ROUTER_REPLAY, api_key=FIREWORKS_API_KEY,
            base_url=_control_plane_base_url(API_BASE_URL),
            additional_headers=None, base_model=active_base_model,
        )
        session_name = getattr(service, "training_session_name", None)
        session_id = getattr(service, "training_session_id", None)
        run_id = getattr(training_client, "run_id", None)
        wandb_run = init_wandb(active_base_model, start_step)
        print(f"Connected: session={session_name or session_id} run={run_id} router_replay={router_replay_enabled}")

        eval_rec = evaluate(training_client, service, start_step)
        append_jsonl(eval_metrics_path, eval_rec); wandb_log(wandb_run, eval_rec, start_step)
        evaluated_at.add(start_step)

        row_cursor = start_step * PROMPT_GROUPS_PER_STEP
        for step in range(start_step, start_step + STEPS):
            batch_indices = [order[(row_cursor + i) % len(order)] for i in range(PROMPT_GROUPS_PER_STEP)]
            row_cursor += PROMPT_GROUPS_PER_STEP
            batch = [train_rows[i] for i in batch_indices]
            rec = training_step(training_client, service, batch, step, router_replay_enabled)
            append_jsonl(metrics_path, rec); wandb_log(wandb_run, rec, step)
            print(f"step {step:02d} reward={rec['rollout/raw_reward']:.3f} filtered={rec['rollout/filtered_reward']:.3f} loss={rec['train/loss']}")
            completed = step + 1
            if DCP_SAVE_INTERVAL and completed % DCP_SAVE_INTERVAL == 0:
                latest_resume_reference = save_dcp_checkpoint(training_client, completed, session_name, run_id)
                saved_dcp_steps.add(completed)
            if EVAL_INTERVAL and completed % EVAL_INTERVAL == 0:
                eval_rec = evaluate(training_client, service, completed)
                append_jsonl(eval_metrics_path, eval_rec); wandb_log(wandb_run, eval_rec, completed)
                evaluated_at.add(completed)

        final_step = start_step + STEPS
        if final_step not in evaluated_at:
            eval_rec = evaluate(training_client, service, final_step)
            append_jsonl(eval_metrics_path, eval_rec); wandb_log(wandb_run, eval_rec, final_step)
        if DCP_SAVE_INTERVAL and final_step not in saved_dcp_steps:
            latest_resume_reference = save_dcp_checkpoint(training_client, final_step, session_name, run_id)
        final_path = training_client.save_weights_for_sampler("countdown-final").result().path
        if not final_path:
            raise RuntimeError("Final sampler checkpoint returned no path")
        (RUN_DIR / "final_checkpoint.txt").write_text(final_path + "\n")
        lifecycle = {
            "session_name": session_name, "session_id": session_id, "run_id": run_id,
            "base_model": active_base_model, "final_sampler_checkpoint": final_path,
            "resume_from": latest_resume_reference or None,
            "start_step": start_step, "completed_steps": final_step,
            "router_replay": router_replay_enabled, "run_profile": RUN_PROFILE,
        }
        (RUN_DIR / "lifecycle.json").write_text(json.dumps(lifecycle, indent=2) + "\n")
        print("Final sampler checkpoint:", final_path)
    finally:
        try:
            if wandb_run is not None:
                wandb_run.finish()
        except Exception as exc:
            print(f"W&B cleanup skipped: {exc}", flush=True)
        finally:
            if service is not None:
                service.close()

## 10. Inspect and download results

Each execution uses a fresh run directory. It contains metrics, the resolved configuration, dataset hash, pinned cookbook revision, exact split, decoded evaluation completions, lifecycle identifiers, and checkpoint references. Full runs save resumable trainer state every five steps. The cell also creates a ZIP; set `DOWNLOAD_RESULTS = True` to download it. Model promotion remains in the upstream command-line example because it is a separate mutating lifecycle operation.

In [ ]:
import shutil

import matplotlib.pyplot as plt

DOWNLOAD_RESULTS = False

metrics_file = RUN_DIR / "metrics.jsonl"
eval_file = RUN_DIR / "eval_metrics.jsonl"
if metrics_file.exists():
    train_metrics = [json.loads(line) for line in metrics_file.open() if line.strip()]
    plt.plot([r["step"] + 1 for r in train_metrics], [r["rollout/raw_reward"] for r in train_metrics], label="rollout")
    if eval_file.exists():
        eval_metrics = [json.loads(line) for line in eval_file.open() if line.strip()]
        plt.plot([r["completed_steps"] for r in eval_metrics], [r["eval/raw_reward"] for r in eval_metrics], marker="o", label="held-out eval")
    plt.xlabel("completed optimizer steps"); plt.ylabel("mean reward"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(RUN_DIR / "reward_curve.png", dpi=160); plt.show()
else:
    print("No completed training metrics; packaging the available diagnostics.")

archive_path = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("Artifacts:", sorted(p.name for p in RUN_DIR.iterdir()))
print("ZIP:", archive_path)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))